In [1]:
import pandas as pd
import numpy as np
from dash import Dash, html, dcc, Input, Output, dash_table
import dash_bootstrap_components as dbc
import plotly.express as px
import matplotlib.pyplot as plt
import plotly.graph_objects as go

In [2]:
farm_files = {
    "Farm 66": "Data/Data_Folder/updated-animal-data-farm-id-66.csv",
    "Farm 68": "Data/Data_Folder/updated-animal-data-farm-id-68.csv",
    "Farm 70": "Data/Data_Folder/updated-animal-data-farm-id-70.csv",
    "Farm 73": "Data/Data_Folder/updated-animal-data-farm-id-73.csv",
    "Farm 74": "Data/Data_Folder/updated-animal-data-farm-id-74.csv",
    "Farm 75": "Data/Data_Folder/updated-animal-data-farm-id-75.csv",
    "New 66": "Data/Extra Stuff/new-animal-data-farm-id-66-2025-10-27-to-2025-11-19.csv"
}

def load_farm_data(farm_name):
   
    df = pd.read_csv(farm_files[farm_name])

    df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'], errors='coerce')

    df['airTemp'] = pd.to_numeric(df.get('airTemp', pd.Series()), errors='coerce')
    df['humidity'] = pd.to_numeric(df.get('humidity', pd.Series()), errors='coerce')

    df['THI'] = (0.8 * df['airTemp'] + 0.01 * df['humidity'] * (df['airTemp'] - 14.4) + 46.4)

    df['THI'] = df['THI'].round(2)

    return df

In [3]:
def countUniqueIDs(selected_farm):
    df = load_farm_data(selected_farm)
    unique_ids = df['id'].unique()
    return len(unique_ids)

In [4]:

from datetime import timedelta

def create_temp_plot(data, color1='#F4D03F', color2='skyblue', color3='lightgreen'):
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=data['datetime'], y=data['temp'], mode='lines',
        name='Calf Temp (°C)', line=dict(color=color1, width=2)
    ))
    fig.add_trace(go.Scatter(
        x=data['datetime'], y=data['airTemp'], mode='lines',
        name='Outside Air Temp (°C)', line=dict(color=color2, width=2, dash='dot')
    ))
    fig.add_trace(go.Scatter(
        x=data['datetime'], y=data['THI'], mode='lines', yaxis='y2',
        name='THI', line=dict(color=color3, width=2, dash='dash')
    ))

    data['month'] = data['datetime'].dt.month

    # --- Summer months ---
    summer_mask = data['month'].isin([5, 6, 7, 8])

    # Heat stress (high THI + high temp)
    heat_stress = data[(summer_mask) & (data['THI'] > 72) & (data['temp'] > 40)]

    # Non-heat fever (summer, normal THI but high temp)
    non_heat_summer = data[(summer_mask) & (data['THI'] <= 72) & (data['temp'] > 40)]

    # Recurrent fever (non-summer)
    non_summer = data[~summer_mask].copy()
    non_summer['date'] = non_summer['datetime'].dt.date
    recurrent_days = (
        non_summer[non_summer['temp'] > 40]
        .groupby('date').filter(lambda x: len(x) > 1)
    )

    # Heat stress fever markers
    fig.add_trace(go.Scatter(
        x=heat_stress['datetime'], y=heat_stress['temp'],
        mode='markers', name='🔥 Heat Stress Fever',
        marker=dict(color='red', size=8, symbol='circle'),
        hovertext=[f"Temp={t:.1f}°C, THI={h:.1f}" for t, h in zip(heat_stress['temp'], heat_stress['THI'])]
    ))

    # Non-heat fever markers
    fig.add_trace(go.Scatter(
        x=non_heat_summer['datetime'], y=non_heat_summer['temp'],
        mode='markers', name='⚠️ Non-Heat Fever (Summer)',
        marker=dict(color='green', size=8, symbol='square'),
        hovertext=[f"Temp={t:.1f}°C, THI={h:.1f}" for t, h in zip(non_heat_summer['temp'], non_heat_summer['THI'])]
    ))

    # Recurrent fever markers
    fig.add_trace(go.Scatter(
        x=recurrent_days['datetime'], y=recurrent_days['temp'],
        mode='markers', name='🩺 Recurrent Fever (Non-Heat)',
        marker=dict(color='blue', size=8, symbol='diamond'),
        hovertext=[f"Temp={t:.1f}°C" for t in recurrent_days['temp']]
    ))

    # --- Event markers ---
    non_null_events = data[data['event'].notnull()]
    if not non_null_events.empty:
        fig.add_trace(go.Scatter(
            x=non_null_events['datetime'],
            y=non_null_events['temp'],
            mode='markers+text',
            name='Events',
            marker=dict(symbol='x', size=12, color='red', line=dict(width=2, color='white')),
            text=non_null_events['event'],
            textposition='top center',
            textfont=dict(color='white', size=10),
            hovertemplate="<b>Event:</b> %{text}<br><b>Date:</b> %{x|%d %b %Y %H:%M}<br><b>Temp:</b> %{y:.1f}°C"
        ))

    # --- Add preclinical window as shaded vertical rectangles (vrect) ---
    for idx, row in non_null_events.iterrows():
        event_datetime = row['datetime']
        start = (event_datetime - timedelta(days=5)).replace(hour=0, minute=0, second=0, microsecond=0)
        end = event_datetime  # exact time of event

        fig.add_vrect(
            x0=start, x1=end,
            fillcolor="orange", opacity=0.15, layer="below", line_width=0,
            annotation_text="Preclinical Window",
            annotation_position="top left",
            annotation_font=dict(color="orange", size=10)
        )

    # --- Layout ---
    fig.update_layout(
        title=f'🐄 Calf ID {data["id"].iloc[0]} — Temperature, Air Temp & THI',
        xaxis_title='Date & Time',
        yaxis_title='Temperature (°C)',
        legend_title='Metrics',
        template='plotly_dark',
        hovermode='x unified',
        yaxis2=dict(title='THI', overlaying='y', side='right', showgrid=False),
        title_font=dict(size=20, family='Arial', color='white')
    )

    return fig



In [ ]:
def create_rumination_plot(df):
    """
    Create an interactive rumination plot for a specific calf.

    Highlights:
    - Ideal rumination range (blue band)
    - Preclinical windows (orange shade)
    - Event markers (red X)
    - Low rumination points inside clinical window (red dots)
    """

    df['date'] = pd.to_datetime(df['date'])

    daily_rumination = df.groupby('date').agg({
        'rumination': 'sum',
        'ageInDays': 'mean'
    }).reset_index()

    daily_rumination['rumination_hours'] = daily_rumination['rumination'] * 0.25

    def ideal_rumination_range(age):
        if pd.isna(age):
            return (None, None)
        elif age < 14:
            return (0, 2)
        elif age < 30:
            return (2, 4)
        elif age < 60:
            return (4, 8)
        else:
            return (6, 12)

    daily_rumination['ideal_min'], daily_rumination['ideal_max'] = zip(
        *daily_rumination['ageInDays'].apply(ideal_rumination_range)
    )


    non_null_events = df[df['event'].notnull()].copy()
    non_null_events['date'] = pd.to_datetime(non_null_events['date'])


    preclinical_days = []
    for d in non_null_events['date']:
        window = pd.date_range(d - pd.Timedelta(days=5), d - pd.Timedelta(days=1))
        preclinical_days.extend(window)
        fig_vrect_color = "orange"
    
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=pd.concat([daily_rumination['date'], daily_rumination['date'][::-1]]),
        y=pd.concat([daily_rumination['ideal_max'], daily_rumination['ideal_min'][::-1]]),
        fill='toself',
        fillcolor='rgba(135, 206, 250, 0.2)',
        line=dict(color='rgba(255,255,255,0)'),
        name='Ideal Range (hrs/day)',
        hoverinfo='skip'
    ))

    fig.add_trace(go.Scatter(
        x=daily_rumination['date'],
        y=daily_rumination['rumination_hours'],
        mode='lines+markers',
        name='Actual Rumination (hrs/day)',
        line=dict(color='lightgreen', width=3, shape='spline'),
        marker=dict(size=8, color='lightgreen'),
        hovertemplate="<b>Date:</b> %{x|%d %b %Y}<br>"
                      "<b>Rumination:</b> %{y:.2f} hrs<br>"
                      "<b>Age:</b> %{customdata:.0f} days",
        customdata=daily_rumination['ageInDays']
    ))

    for d in non_null_events['date']:
        start = d - pd.Timedelta(days=5)
        end = d - pd.Timedelta(days=1)
        fig.add_vrect(
            x0=start, x1=end,
            fillcolor="orange", opacity=0.15, layer="below", line_width=0,
            annotation_text="Preclinical Window",
            annotation_position="top left",
            annotation_font=dict(color="orange", size=10)
        )

    # --- Identify low rumination in clinical window ---
    daily_rumination['in_window'] = daily_rumination['date'].isin(preclinical_days)
    daily_rumination['below_range'] = daily_rumination['rumination_hours'] < daily_rumination['ideal_min']

    low_points = daily_rumination[daily_rumination['in_window'] & daily_rumination['below_range']]

    # --- Add low rumination markers (inside clinical window) ---
    if not low_points.empty:
        fig.add_trace(go.Scatter(
            x=low_points['date'],
            y=low_points['rumination_hours'],
            mode='markers',
            name='Low Rumination (Preclinical)',
            marker=dict(size=10, color='red', symbol='circle', line=dict(width=1, color='white')),
            hovertemplate="<b>Date:</b> %{x|%d %b %Y}<br>"
                          "<b>Rumination:</b> %{y:.2f} hrs<br>"
                          "<b>Status:</b> Below Ideal (Preclinical)"
        ))

    # --- Event markers (red X) ---
    fig.add_trace(go.Scatter(
        x=non_null_events['date'],
        y=[
            daily_rumination.set_index('date').loc[d, 'rumination_hours']
            if d in daily_rumination['date'].values else None
            for d in non_null_events['date']
        ],
        mode='markers+text',
        name='Events',
        marker=dict(symbol='x', size=12, color='red', line=dict(width=2, color='white')),
        text=non_null_events['event'],
        textposition='top center',
        textfont=dict(color='white', size=12),
        hovertemplate="<b>Event:</b> %{text}<br><b>Date:</b> %{x|%d %b %Y}"
    ))

    fig.update_layout(
        title=f"🐄 Daily Rumination Pattern — Calf ID {df['id'].iloc[0]}",
        xaxis_title="Date",
        yaxis_title="Rumination (hours per day)",
        template="plotly_dark",
        hovermode="x unified",
        legend=dict(title="Legend", orientation="h", y=-0.25, x=0.3),
        title_font=dict(size=22, family="Arial", color="white"),
        xaxis=dict(showgrid=False),
        yaxis=dict(showgrid=True, gridcolor="gray"),
        margin=dict(l=60, r=30, t=80, b=80)
    )

    return fig


In [ ]:
# def create_activity_plot(df):
#     """
#     df: pandas DataFrame filtered for one calf and date range, must have
#         'date', 'accel', and 'ageInDays' columns.

#     Returns: Plotly Figure object with daily activity and ideal range.
#     """

#     # Aggregate daily activity and average age
#     daily_activity = df.groupby('date').agg({
#         'accel': 'mean',
#         'ageInDays': 'mean'
#     }).reset_index()

#     # Define ideal activity range depending on age
#     def ideal_activity_range(age):
#         if age < 9:
#             return (0.05, 0.1)
#         elif age < 15:
#             return (0.15, 0.25)
#         else:
#             return (0.13, 0.2)

#     daily_activity['ideal_min'], daily_activity['ideal_max'] = zip(
#         *daily_activity['ageInDays'].apply(ideal_activity_range)
#     )

#     fig = go.Figure()

#     # Actual daily activity
#     fig.add_trace(go.Scatter(
#         x=daily_activity['date'],
#         y=daily_activity['accel'],
#         mode='lines+markers',
#         name='Actual Activity (avg accel)',
#         line=dict(color='orange', width=3),
#         marker=dict(size=8),
#         hovertemplate="<b>Date:</b> %{x}<br>" +
#                       "<b>Activity:</b> %{y:.2f}<br>" +
#                       "<b>Age:</b> %{customdata} days",
#         customdata=daily_activity['ageInDays']
#     ))

#     # Ideal activity range band
#     fig.add_trace(go.Scatter(
#         x=pd.concat([daily_activity['date'], daily_activity['date'][::-1]]),
#         y=pd.concat([daily_activity['ideal_max'], daily_activity['ideal_min'][::-1]]),
#         fill='toself',
#         fillcolor='rgba(255, 165, 0, 0.15)',  # light orange transparent
#         line=dict(color='rgba(255,255,255,0)'),
#         name='Ideal Range (avg accel)',
#         hoverinfo='skip'
#     ))

#     fig.update_layout(
#         title=f"Daily Activity Pattern — Calf ID {df['id'].iloc[0]}",
#         xaxis_title="Date",
#         yaxis_title="Activity (average accel)",
#         template="plotly_dark",
#         hovermode="x unified",
#         legend=dict(title="Legend", orientation="h", y=-0.2, x=0.3),
#         title_font=dict(size=22, family="Arial", color="white"),
#         xaxis=dict(showgrid=False),
#         yaxis=dict(showgrid=True, gridcolor="gray")
#     )

#     return fig


In [7]:
def eventTable(selected_farm, selected_id):
    df = load_farm_data(selected_farm)
    df = df[df['id'] == selected_id]

    events = df[df['event'].notnull()].copy()

    events['datetime'] = pd.to_datetime(events['datetime'])


    events['date'] = events['datetime'].dt.strftime('%#d %B, %Y')  
    events['time'] = events['datetime'].dt.strftime('%H:%M:%S')   

    events['date'] = events['date'].astype(str)
    events['time'] = events['time'].astype(str)

    table_data = events[['id', 'event', 'date', 'time', 'temp', 'THI']].rename(
        columns={'temp': 'body_Temperature'}
    )

    return table_data


In [8]:
app = Dash(__name__, suppress_callback_exceptions=True)

label_style = {
    "fontWeight": "bold",
    "fontSize": "16px",
    "marginRight": "10px"
}

app.layout = html.Div([
    html.H2("🐄 Farm Data EDA Dashboard", style={"textAlign": "center", "marginBottom": "30px"}),

    html.Div([
        # Select Farm
        html.Div([
            html.Label("Select Farm:", style=label_style),
            dcc.Dropdown(
                id='farm-dropdown',
                options=[{'label': name, 'value': name} for name in farm_files.keys()],
                value='Farm 68',
                clearable=False,
                style={'width': '100px'}
            ),
        ], style={'display': 'inline-block', 'verticalAlign': 'top', 'marginRight': '40px'}),

        # Select Calf ID
        html.Div([
            html.Label("Select Calf ID:", style=label_style),
            dcc.Dropdown(
                id='id-dropdown',
                placeholder="Select a calf...",
                value=2293,
                style={'width': '100px'}
            ),
        ], style={'display': 'inline-block', 'verticalAlign': 'top', 'marginRight': '40px'}),

        # Date Picker
        html.Div(
            id='date-picker-container',
            style={'display': 'inline-block', 'verticalAlign': 'top', 'marginRight': '200px'}
        ),

        # Unique ID Count Display
        html.Div(
            id='unique-id-count',
            style={'display': 'inline-block', 'verticalAlign': 'top-right', 'fontSize': '24px', 'color': 'black'}
        )
    ], style={'textAlign': 'left', 'marginBottom': 30}),

    html.Div([
        dash_table.DataTable(
                id='event-table',
                columns=[
                    {"name": "id", "id": "id"},
                    {"name": "event", "id": "event"},
                    {"name": "date", "id": "date"},
                    {"name": "time", "id": "time"},
                    {"name": "body_Temperature", "id": "body_Temperature"},
                    {"name": "THI", "id": "THI"}
                ],
                style_table={'overflowX': 'auto', 'marginBottom': '30px'},
                style_cell={'textAlign': 'center', 'padding': '6px'},
                style_header={'backgroundColor': '#0d1117', 'color': 'white', 'fontWeight': 'bold'},
                style_data={'backgroundColor': '#161b22', 'color': 'white'},
                page_size=5
            ),
    ], style={'margin': '20px'}),

    dcc.Graph(id='temp-plot', style={'height': '600px', 'marginTop': '30px'}),
    dcc.Graph(id='rumination-plot', style={'height': '600px', 'marginTop': '30px'}),
    dcc.Graph(id='activity-fig', style={'height': '600px', 'marginTop': '30px'})
])

@app.callback(
    Output('id-dropdown', 'options'),
    Input('farm-dropdown', 'value')
)
def update_id_dropdown(selected_farm):
    df = load_farm_data(selected_farm)
    calf_ids = df['id'].unique()
    return [{'label': str(i), 'value': i} for i in calf_ids]

@app.callback(
    Output('date-picker-container', 'children'),
    Input('id-dropdown', 'value'),
    Input('farm-dropdown', 'value')
)
def show_date_picker(selected_id, selected_farm):
    if not selected_id:
        return "" 

    # Load and preprocess farm data
    df = load_farm_data(selected_farm)
    df_calf = df[df['id'] == selected_id]

    # Handle empty selection gracefully
    if df_calf.empty or df_calf['datetime'].isna().all():
        return html.Div("⚠️ No valid date data available for this calf.")

    # Get min and max available dates for this calf
    min_date = df_calf['datetime'].min().date()
    max_date = df_calf['datetime'].max().date()

    return html.Div([
        html.Label("Select Date Range:", style=label_style),
        dcc.DatePickerRange(
            id='date-picker',
            start_date=min_date,
            end_date=max_date,
            min_date_allowed=min_date,
            max_date_allowed=max_date,
            display_format='MMM D, YYYY',
            style={'display': 'inline-block'}
        )
    ])

@app.callback(
    Output('unique-id-count', 'children'),
    Input('farm-dropdown', 'value')
)
def update_unique_id_count(selected_farm):
    if not selected_farm:
        return ""
    count = countUniqueIDs(selected_farm)
    return f"📊 Total Unique Calf IDs in {selected_farm}: {count}"

@app.callback(
    Output('temp-plot', 'figure'),
    Input('farm-dropdown', 'value'),
    Input('id-dropdown', 'value'),
    Input('date-picker', 'start_date'),
    Input('date-picker', 'end_date'),
    prevent_initial_call=True
)
def update_temp_plot(selected_farm, selected_id, start_date, end_date):
    if not all([selected_farm, selected_id, start_date, end_date]):
        return go.Figure()

    df = load_farm_data(selected_farm)
    df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
    df = df[df['id'] == selected_id].copy()

    mask = (df['datetime'] >= pd.to_datetime(start_date)) & (df['datetime'] <= pd.to_datetime(end_date))
    df = df[mask]

    if df.empty:
        fig = go.Figure()
        fig.update_layout(title="No data available for the selected date range", template='plotly_dark')
        return fig

    return create_temp_plot(df)

@app.callback(
    Output('rumination-plot', 'figure'),
    Input('farm-dropdown', 'value'),
    Input('id-dropdown', 'value'),
    Input('date-picker', 'start_date'),
    Input('date-picker', 'end_date')
)
def update_rumination_plot(selected_farm, selected_id, start_date, end_date):
    if not all([selected_farm, selected_id, start_date, end_date]):
        return go.Figure()

    df = load_farm_data(selected_farm)
    df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    
    df = df[df['id'] == selected_id].copy()
    mask = (df['date'] >= pd.to_datetime(start_date)) & (df['date'] <= pd.to_datetime(end_date))
    df = df[mask]

    if df.empty:
        fig = go.Figure()
        fig.update_layout(
            title="No rumination data available for the selected filters",
            template='plotly_dark'
        )
        return fig

    return create_rumination_plot(df)

@app.callback(
    Output('activity-fig', 'figure'),
    Input('farm-dropdown', 'value'),
    Input('id-dropdown', 'value'),
    Input('date-picker', 'start_date'),
    Input('date-picker', 'end_date'),
    prevent_initial_call=True
)
def update_activity_plot(selected_farm, selected_id, start_date, end_date):
    if not all([selected_farm, selected_id, start_date, end_date]):
        return go.Figure()

    df = load_farm_data(selected_farm)
    df['datetime'] = pd.to_datetime(df.get('datetime'), errors='coerce')
    # Ensure 'date' exists
    if 'date' not in df.columns:
        df['date'] = df['datetime'].dt.date
    df['date'] = pd.to_datetime(df['date'], errors='coerce')

    
    df['id'] = df['id'].astype(str)
    selected_id = str(selected_id)
    df = df[df['id'] == selected_id].copy()

    mask = (df['date'] >= pd.to_datetime(start_date)) & (df['date'] <= pd.to_datetime(end_date))
    df = df[mask]

    if df.empty:
        fig = go.Figure()
        fig.update_layout(
            title="No activity data available for the selected filters",
            template='plotly_dark'
        )
        return fig

    return create_activity_plot(df)

@app.callback(
    Output('event-table', 'data'),
    Input('farm-dropdown', 'value'),
    Input('id-dropdown', 'value')
)
def update_event_table(selected_farm, selected_id):
    if not selected_farm or not selected_id:
        return []
    df_table = eventTable(selected_farm, selected_id)
    return df_table.to_dict('records')

if __name__ == '__main__':
    app.run(port=5000, debug=True)

C:\Users\rashe\AppData\Local\Temp\ipykernel_2008\3921853781.py:13: DtypeWarning:

Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.

C:\Users\rashe\AppData\Local\Temp\ipykernel_2008\3921853781.py:13: DtypeWarning:

Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.

C:\Users\rashe\AppData\Local\Temp\ipykernel_2008\3921853781.py:13: DtypeWarning:

Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.

C:\Users\rashe\AppData\Local\Temp\ipykernel_2008\3921853781.py:13: DtypeWarning:

Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.

C:\Users\rashe\AppData\Local\Temp\ipykernel_2008\3921853781.py:13: DtypeWarning:

Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.

C:\Users\rashe\AppData\Local\Temp\ipykernel_2008\3921853781.py:13: DtypeWarning:

Columns (1) have mixed types. Specify dtype option on import or set low_m